# Finite state automata

## Deterministic finite-state automaton

A deterministic finite-state automaton (DFA) is a *finite-state machine* (a mathematical model of computation) that accepts or rejects a given string of symbols by progressing through a state sequence uniquely defined by the string. 

Formally, a DFA is defined as a 5-tuple  ($Q, \Sigma, \delta, q_0, F$) where:
 - $Q$ is a finite set of states
 - $\Sigma$ is a finite alphabet
 - $\delta : Q \times \Sigma \to Q$ is the transition function
 - $q_0 \in Q$ is the start state
 - $F \subseteq Q$ is the set of accept states

![DFA example](dfa_example.png)

*This DFA example accepts the strings ending in 1, with:*

 - $Q = \{q_1, q_2\}$,
 - $\Sigma = \{0,1\}$, 
 - $\delta(q_1,0) = q_1,\quad \delta(q_1,1) = q_2,\quad \delta(q_2,0) = q_1,\quad \delta(q_2,1) = q_2$,
 - $q_0 = q_1$,
 - $F = \{q_2\}$



## Regular languages

A language is simply a set of strings over an alphabet.

The DFA $M$ recognises the language $L$ if $L = \{\, w \mid M \text{ accepts } \,w\}$. 
 - The language $L$ is the set of all strings that the DFA $M$ accepts

A regular language is any language that can recognised by some DFA.
 - If you can build a DFA that accepts exactly the strings in that language, then that language is regular

### Regular expressions

Regular expressions are another way of describing regular languages, using algebra-like rules instead of state diagrams. Regular expressions and DFAs have exactly the same expressive power.

Regular expressions are built using some basic operations, called regular operations, on languages:
 - Boolean (set-theoretic) operations - general set operations can be applied to languages, since languages are sets of strings
   - Union: $A \cup B = \{\, x \mid x \in A \text{ or } x \in B \,\}$
   - Intersection: $A \cap B = \{\, x \mid x \in A \text{ and } x \in B \,\}$
   - Difference: $A \setminus B = \{\, x \mid x \in A \text{ and } x \notin B \,\}$
   - Complement: $\overline{A} = \Sigma^{*} \setminus A$
 - Language theory specific operations - special to formal languages and regular expressions
   - Concatenation: $A \circ B = \{\, xy \mid x \in A \text{ and } y \in B \,\}$
   - Star: $A^{*} = \{\, x_1 x_2 \dots x_k \mid k \ge 0 \text{ and } x_i \in A \text{ for every } i,\ 1 \le i \le k \,\} = \bigcup_{i=0}^{\infty} A^{i}$
     - in other words, $A^{*}$ is the set of all finite strings obtained by concatenating zero or more strings from $A$, allowing repetition

The formal definition of a regular expression (RE) is inductive (recursive) - there are initial REs, and new REs can be obtained from old ones by means of regular operations.

$R$ is a regular expression over the alphabet $\Sigma$ if $R$ is any of:
1. $a$, for some $a \in \Sigma$,
2. $\varepsilon$ (the empty string),
3. $\emptyset$ (the empty language),
4. $(R_1 \cup R_2)$, where $R_1$ and $R_2$ are regular expressions,
5. $(R_1 \circ R_2)$, where $R_1$ and $R_2$ are regular expressions, and
5. $(R_1^{*})$, where $R_1$ is a regular expression.

Note:
- the concatenation symbol can be omitted, i.e. $R_1 R_2$ means $R_1 \circ R_2$
- parenthesis can be omitted, bearing in mind the precedence order from highest to lowest is $^*$, $\circ$, $\cup$, i.e. $01 \cup 1^*$ means $((01) \cup (1^{*}))$

## Combining automata

Automata can be combined in order to implement an operation on languages. This is to build a new DFA that recognises a new language, specifically one obtained by applying an operation (e.g. union) to languages that are already recognised by DFAs.

For example, given $L_1$ is recognised by $M_1 = (Q_1, \Sigma, \delta_1, q_1, F_1)$ and $L_2$ is recognised by $M_2 = (Q_2, \Sigma, \delta_2, q_2, F_2)$, we can combine $M_1$ and $M_2$ into a new automaton M that would recognise $L_1 \cup L_2$.

It will not work to first simulate the input on $M_1$, then simulate it on $M_2$, and accept if either $M_1$ or $M_2$ or both accept, since after $M_1$ has run on the input, the input is exhausted - A DFA cannot 'rewind' the input in order to run $M_2$.

The solution is to run $M_1$ and $M_2$ in parallel, using pairs of states (a state for each DFA). The transition function can be defined as below:
![Transition function](parallel_transition_function.png)

Formally, the components of the new automaton $M = (Q, \Sigma, \delta, q_0, F)$ are:
- $Q = Q_1 \times Q_2$
- $\delta((s,u), a) = (\delta_1(s,a), \delta_2(u,a)), \quad \forall s \in Q_1, u \in Q_2, a \in \Sigma$
- $q_0 = (q_1, q_2)$
- $F = \{ (s,u) \in Q \mid s \in F_1 \text{ or } u \in F_2 \}$

## Concatenation

The above method for combining automata doesn't work for recognising concatenation. 

$w \in L_1 \circ L_2$ only if there are words $x$ and $y$ such that $w = xy$, $x \in L_1$ and $y \in L_2$.

If $L_1$ is recognised by $M_1 = (Q_1, \Sigma, \delta_1, q_1, F_1)$ and $L_2$ is recognised by $M_2 = (Q_2, \Sigma, \delta_2, q_2, F_2)$, then to recognise $w$, $M_1$ would need to run on some prefix of $w$, and $M_2$ on the rest. The problem lies in finding the break point for this - the only way to do this is to guess it non-deterministically. 

In order to implement concatenation, non-deterministic finite automatons are required.

# Non-deterministic Finite Automaton (NFA)

A nondeterministic finite-state automaton (NFA) is a finite-state machine that accepts or rejects a given string of symbols by progressing through one or more possible state sequences defined by the string, where at least one such sequence leads to an accepting state.

Formally, an NFA is defined as a 5-tuple  ($Q, \Sigma, \delta, q_0, F$) where:
 - $Q$ is a finite set of states
 - $\Sigma$ is a finite alphabet
 - $\delta : Q \times (\Sigma \cup \{\varepsilon\}) \to \mathcal{P}(Q)$ is the transition function
 - $q_0 \in Q$ is the start state
 - $F \subseteq Q$ is the set of accept states

The only difference between the DFA and NFA definitions is in the transition function:
- in a DFA, the transition function maps a state and an input symbol to exactly one next state
- in an NFA, the transition function the transition function maps a state and an input symbol (or $\varepsilon$) to a set of possible next states, allowing multiple or zero transitions. Here, $\varepsilon$ represents the empty string, meaning the automaton can move to the next state without consuming any input symbol

![NFA example](nfa_example.png)

From this example, three potential problems can be identified:
- there is more than one next possible state (in $q_1$, an input $1$ can lead to $q_1$ or $q_2$)
- $\varepsilon$-transition (in $q_2$, if the next symbol is $0$, and $\varepsilon$ is used to transition to $q_3$, then the $0$ will not be consumed, which may lead to rejection, even though the string may contain the substring $101$)
- no 'next' state (in $q_3$, an input 0 has no next state, so the computation hangs, meaning it ends before the input has been exhausted)
  - in this case, this is treated as rejection
 
These are not actually problems, due to the nature of how NFAs work:
- an NFA can have multiple possible next states for the same input symbol and $\varepsilon$-transitions that move without consuming input
- conceptually, the NFA explores all possible paths simultaneously
- the input is accepted if any path reaches an accepting state after consuming the entire input
- if all paths fail, the input is rejected

Since every possibility is tried: 
- multiple next states are fine - each path is explored
- $\varepsilon$-transitions are fine - there's also a path where the $\varepsilon$-transition isn't taken, so the input isn't lost
- paths that 'hang' (no next state) are fine - other valid paths will continue and may still lead to acceptance

Using the NFA example above:

![All possible computation](nfa_all_computations.png)

### Computation of an NFA

Let $M = (Q, \Sigma, \delta, q_0, F)$ be an NFA and $w = w_1 w_2 \dots w_n$ be a word over $\Sigma \cup \{\varepsilon\}$.

$M$ accepts $w$ if there exists a sequence of states $r_0, r_1, r_2, \dots, r_n$ satisfying the following conditions:
- $r_0 = q_0$
- $r_{i+1} \in \delta(r_i, w_{i+1})$ for every $i$, $0 \le i \le n-1$
- $r_n \in F$

### DFA vs NFA computation
- In DFA, there is a unique computation (path). It either accepts or rejects.
- In NFA, there are, in general, many computational paths. The NFA accepts if at least one computation accepts. The NFA rejects only if all the computations reject.

## Converting an NFA into a DFA

In an NFA, at any point, the machine could be in multiple possible states at the same time because of non-determinism. A DFA cannot be in multiple states at once - it must have exactly one current state. Therefore, in order to convert an NFA into a DFA, make each state of the DFA represent a subset of states of the NFA.

In the converted DFA:
- each DFA state is a set of NFA states
- the DFA start state is the set containing the NFA start state and all states reachable from it via $\varepsilon$-transitions
- the DFA transition function is defined as:
  - For a DFA state $R \subseteq Q_{NFA}$ and input symbol $a \in \Sigma$:
    1. Look at all NFA states in $R$
    2. For each state $r \in R$, see where the NFA can go on input $a$
    3. Take the union of all these resulting states
    4. Add all states reachable via $\varepsilon$-transitions
       - This gives the next DFA state
- a DFA state is accepting if it contains any NFA accepting state

An example of how a the transition function of the DFA can be defined is given below:

![NFA TO DFA transition function definition](nfa_dfa_trans_func.png)

More formally, let $N = (Q, \Sigma, \delta, q_0, F)$ be an NFA recognising some language $L$. We construct a DFA $M = (Q', \Sigma, \delta', q_0', F')$ recognising the same language $L$ as follows:

Define the $\varepsilon$-closure of a set of states:
$$ E(R) = \{ q \in Q \mid q \text{ is reachable from some } r \in R \text{ through a number of $\varepsilon$-transitions} \}. $$
(Note that $R \subseteq E(R)$ because each state is reachable from itself using zero $\varepsilon$-transitions.)

1. $Q' = \mathcal{P}(Q)$, the powerset (all possible subsets) of $Q$.
2. For $R \in Q'$ and $a \in \Sigma$, define
    $$ \delta'(R, a) = \bigcup_{r \in R} E(\delta(r,a)) = \{ q \in Q \mid q \in E(\delta(r,a)) \text{ for some } r \in R \}$$
3. $q_0' = E(\{q_0\})$.
4. $F' = \{ R \in Q' \mid R \cap F \neq \emptyset \}$.

An example of the conversion of an NFA to a DFA is shown below:

![NFA to DFA conversion](nfa_to_dfa.png)

Note that the red states are redundant and can be removed, as there are no ways to reach those states from the start state. The {} state is necessary as it collects all the paths that would 'hang' in the NFA. 